In [ ]:
# Importiamo le librerie necessarie
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

# Pre-processiamo le immagini
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train[..., np.newaxis]  # (60000, 28, 28, 1)
x_test  = x_test[..., np.newaxis]   # (10000, 28, 28, 1)

# Costruiamo il CNN
model = models.Sequential()

# Step 1: Creiamo 32 filtri 3x3 che scorrono attraverso l'immagine
model.add(layers.Conv2D(filters=32, kernel_size=3, padding='same', activation='relu', input_shape=(28,28,1)))

# Step 2: Consideriamo i quadrati 2x2 che compongono l'immagine
# e consideriamo il valore maggiore, riducendo le dimensioni
model.add(layers.MaxPooling2D(pool_size=2, strides=2))

# Step 3: Dalla griglia bidimensionale di valori, la "spalmiamo" in un vettore
model.add(layers.Flatten())

# Step 4: Rete Feed-Forward che processa il vettore dell'ultimo step
model.add(layers.Dense(units=128, activation='relu'))

# Step 5: Aggiungiamo lo strato finale del modello, un vettore di dimensione 10
# che tramite il softmax diventa una distribuzione di probabilità sulle possibili cifre
model.add(layers.Dense(units=10, activation='softmax'))

# Definiamo un learning rate dinamico, inizialmente a 0.01 con una diminuzione
# quando con il proseguire dell'addestramento si richiede un fine-tuning con un
# aggiornamento dei pesi meno drastico
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=10000,
    decay_rate=0.9)

# Definiamo l'ottimizzatore (Stochastic Gradient Descent)
opt = tf.keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.5)

def explicit_loss(y_true, y_pred):
    # Step 1: Estraiamo la probabilità stimata dalla rete che il valore sia quello corretto
    correct_class_prob = tf.gather(y_pred, tf.cast(y_true, tf.int32), batch_dims=1)

    # Step 2: Applichiamo il logaritmo negativo
    loss = -tf.math.log(correct_class_prob + 1e-7)  # +1e-7 evita di avere log(0)

    return tf.reduce_mean(loss)


def explicit_accuracy(y_true, y_pred):
    # Step 1: Stabiliamo la cifra determinata dal modello
    predicted_class = tf.argmax(y_pred, axis=1)

    # Step 2: Confrontiamola con la cifra corretta
    correct_predictions = tf.equal(tf.cast(y_true, tf.int64), predicted_class)

    # Step 3: Calcoliamo la percentuale di previsioni corrette
    accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))

    return accuracy

model.compile(
    optimizer=opt,
    loss=explicit_loss,
    metrics=[explicit_accuracy]
)

# Riassunto dell'architettura del modello
model.summary()

def explicit_model_fit(model, X, y, epochs=40, batch_size=64, val_split=0.1, verbose=True):
    # Randomizziamo l'ordine dei dati
    p = np.random.permutation(len(X))
    cut = int(len(X) * (1 - val_split))
    train_indices, val_indices = p[:cut], p[cut:]

    Xt, yt = X[train_indices], y[train_indices]
    Xv, yv = X[val_indices], y[val_indices]

    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    optimizer = model.optimizer # Usiamo l'ottimizzatore già compilato nel modello

    for e in range(epochs):
        epoch_losses = []
        epoch_accuracies = []

        # Mescoliamo i dati di training ad ogni epoca per ridurre il rischio di overfitting
        epoch_p = np.random.permutation(len(Xt))
        Xt_shuffled = Xt[epoch_p]
        yt_shuffled = yt[epoch_p]

        for i in range(0, len(Xt_shuffled), batch_size):
            X_batch = Xt_shuffled[i:i + batch_size]
            y_batch = yt_shuffled[i:i + batch_size]

            with tf.GradientTape() as tape:
                # "Forward pass", il modello fa una predizione sulla cifra di input
                y_pred = model(X_batch, training=True)
                # Calcoliamo loss e accuracy delle previsioni
                loss = explicit_loss(y_batch, y_pred)
                acc = explicit_accuracy(y_batch, y_pred)

            # Calcoliamo il gradiente
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

            epoch_losses.append(loss.numpy())
            epoch_accuracies.append(acc.numpy())

        # Calcoliamo la media della loss e accuracy per l'epoca
        history['loss'].append(np.mean(epoch_losses))
        history['accuracy'].append(np.mean(epoch_accuracies))

        # Usiamo model.evaluate sul validation set
        val_loss, val_acc = model.evaluate(Xv, yv, batch_size=batch_size, verbose=0)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        if verbose:
            print(f"Epoch {e+1}/{epochs} | "
                  f"loss {history['loss'][-1]:.4f} acc {history['accuracy'][-1]:.4f} | "
                  f"val_loss {history['val_loss'][-1]:.4f} val_acc {history['val_accuracy'][-1]:.4f}")

    return history

# Eseguiamo il training con la nostra funzione esplicita
history = explicit_model_fit(model, x_train, y_train, epochs=40, batch_size=64, val_split=0.1, verbose=1)

# Creiamo i grafici della loss function e accuracy function
for metric in ('accuracy', 'loss'):
    plt.plot(history[metric], label=f'train {metric}')
    plt.plot(history[f'val_{metric}'], label=f'val {metric}')
    plt.title(f'Model {metric.capitalize()}')
    plt.xlabel('Epoch')
    plt.ylabel(metric.capitalize())
    plt.legend()
    plt.grid(True)
    plt.show()

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest loss  : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Salvataggio del modello
model.save('cnn_model.h5')
print("Modello salvato come 'cnn_model.h5'")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 804,554 (3.07 MB)

 Trainable params: 804,554 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

KeyboardInterrupt: 